# 3. Prepare the Data

Owner: Mia Scalabrin

Cleaning, feature engineering, scaling, encoding and the leak-free preprocessing pipeline. Training period only (orders bought before 1 April 2018). The pipeline built here is reused unchanged in notebooks 04a, 04b and 04c.

## 0. Setup

Starting from the team's final table (85,371 orders), the training period is orders purchased before 1 April 2018 (64,038 orders, split decided by the team, DECISIONS D6). 

Section 3 only uses the training data. The test set (orders from April 2018 onward) is not touched here and is evaluated at the end of the project.

The model inputs are the 22 approval-time features. IDs are excluded because they carry no meaning, raw dates are excluded because their information is already in the engineered timing features (plus the delivery date would be a data leakage), and is_late is of course excluded as this is the target variable

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/olist_final.csv",
                 parse_dates=["order_purchase_timestamp"])
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Training period only: orders purchased before 1 April 2018 (the team's split)
train = df[df["order_purchase_timestamp"] < "2018-04-01"]

# Features = everything except IDs, raw dates and the target
non_features = ["order_id", "customer_id", "order_purchase_timestamp", "order_approved_at",
                "order_delivered_customer_date", "order_estimated_delivery_date", "is_late"]
X_train = train.drop(columns=non_features)
y_train = train["is_late"]
print(X_train.shape)

(64038, 22)


## 3.1 Cleaning

### 3.1a Missing Values

Five columns have missing values in the 64,038 training orders (primary product category, max distance, mean distance, max product weight and mean product weight). 

Of these 5 columns, the product category column has the most missing values (about 1.83% rows total). The distance columns are both missing about 0.52% of rows, and the product weight columns are missing only about 0.02%.

In [2]:
missing = X_train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({"rows_missing": missing,
              "percent": (missing / len(X_train) * 100).round(2)})

,rows_missing,percent
primary_product_category,1169,1.83
max_distance_km,330,0.52
mean_distance_km,330,0.52
mean_product_weight_g,16,0.02
max_product_weight_g,16,0.02


Dropping rows would remove real orders, so we are filling the missing rows instead of removing them. The empty rows in the numerical columns are replaced with the median values, and the empty rows in the category column are replaced with "unknown". 

**The below step is shown on a copy of the table. The same filling is applied inside the pipeline (3.4), so it is fit on the training data only.**

Additionally, the distance columns will also get a "was missing" flag in the pipeline (3.4). This is because a missing distance means the ZIP code was not found in the original geolocation table, which might point to strange addresses (and could contribute to why an order was delivered late).

In [3]:
dist_cols = ["mean_distance_km", "max_distance_km"]
weight_cols = ["mean_product_weight_g", "max_product_weight_g"]

X_train_filled = X_train.copy()

for c in dist_cols:
    X_train_filled[c + "_was_missing"] = X_train_filled[c].isna().astype(int)

fill_cols = dist_cols + weight_cols
X_train_filled[fill_cols] = X_train_filled[fill_cols].fillna(X_train_filled[fill_cols].median())
X_train_filled["primary_product_category"] = X_train_filled["primary_product_category"].fillna("unknown")

print(X_train_filled.isna().sum().sum())

0


### 3.1b Outliers

The below table shows which columns have extreme values (potential outliers).

In [4]:
X_train.select_dtypes("number").skew().sort_values(ascending=False).round(1)

total_price                    11.1
seller_count                   10.7
total_freight                   8.7
item_count                      8.2
distinct_product_count          7.7
total_product_volume_cm3        7.7
approval_lag_hours              5.2
freight_to_price_ratio          3.7
mean_product_weight_g           3.6
max_product_weight_g            3.6
mean_distance_km                1.7
max_distance_km                 1.7
promised_days_from_approval     1.3
customer_zip_code_prefix        0.7
same_state_item_share           0.7
approval_month                  0.2
purchase_month                  0.2
purchase_weekday                0.1
purchase_hour                  -0.6
dtype: float64

#### No outliers are removed
We did not remove any outliers because the extreme values are all still real orders (i.e., 0 rows removed), so skewed columns are handled with a log transform and scaling instead.

The data exploration (notebook 02) did find a few values that are not real: 3 distances above 4,500 km (geocoding errors) and 16 product volumes stored as 0 (missing product dimensions). Together they are 19 of the 64,038 training orders (0.03%), too few to change the results, so they were left as they are. README section 7 lists this as a limitation.

In [5]:
cols = ["total_price", "seller_count", "total_freight", "item_count",
        "distinct_product_count", "total_product_volume_cm3", "approval_lag_hours",
        "freight_to_price_ratio", "mean_product_weight_g", "max_product_weight_g"]

summary = X_train[cols].agg(["median", lambda s: s.quantile(0.99), "max"]).T
summary.columns = ["median", "99th percentile", "max"]
summary.round(1)


,median,99th percentile,max
total_price,85.0,978.0,13440.0
seller_count,1.0,2.0,5.0
total_freight,16.7,98.0,1002.3
item_count,1.0,3.0,21.0
distinct_product_count,1.0,2.0,7.0
total_product_volume_cm3,7744.0,140448.0,1476000.0
approval_lag_hours,0.3,87.4,741.4
freight_to_price_ratio,0.2,1.4,5.7
mean_product_weight_g,700.0,19100.0,40425.0
max_product_weight_g,700.0,19400.0,40425.0


### 3.1c Duplicates
There are no duplicate orders (every order_id in the 64,038 training orders appears only once) therefore 0 rows are removed

In [6]:
print(train["order_id"].is_unique)
print(train["order_id"].duplicated().sum())

True
0


## 3.2 Feature Engineering

### 3.2a Feature selection
Two features are dropped from the current features because they do not provide as much useful information: customer_zip_code_prefix and approval_month

customer_zip_code_prefix has 13,584 different values. One-hot encoding them would turn each value into its own column, which would make the table far too big and slow. As well, Customer state and distance columns already describe the locations.

approval_month is the same as purchase_month for 98.2% of orders. As well, the approval timing is already accounted for by the columns approval_lag_hours and promised_days_from_approval. 

This leaves 20 features instead of 22.

In [7]:
print(X_train["customer_zip_code_prefix"].nunique())
print((X_train["approval_month"] == X_train["purchase_month"]).mean().round(3))

drop_cols = ["customer_zip_code_prefix", "approval_month"]
X_train_sel = X_train.drop(columns=drop_cols)
print(X_train_sel.shape)

13584
0.982
(64038, 20)


### 3.2b Feature groups

| Group | Features | Why it should help predict late deliveries |
|---|---|---|
| Timing | promised days, approval lag, purchase month, weekday and hour | Short promises, slow approvals and busy periods make delays more likely |
| Location | customer state, seller state | Some regions may be harder to deliver to |
| Distance | mean and max distance, same-state share | The farther a package travels, the longer it takes |
| Order size and cost | item count, seller count, total price, freight, freight-to-price ratio | Larger orders and orders from multiple sellers may be harder to ship on time |
| Product | weight, volume, category | Heavy or bulky products are slower to handle and ship |

### 3.2c Transformations
Discretizing continuous features is not necessary because the models can use the raw numbers. As well, date and time features are already split into month, weekday and hour. 

The 13 columns with skew above 1 get a log(1 + x) transform, which makes the distribution more even by compressing the largest values (shrinks the long tails).

We note that log-transforming 3 of these columns did not help much (seller_count, distinct_product_count and item_count). However, keeping them logged is harmless and keeps the treatement consistent. 

As well, for the two distance columns, the log transform over-corrects the skew, from +1.7 to −1.1. The skew is still lower than before, so the log transform is kept for these columns.

In [8]:
import numpy as np

log_cols = ["total_price", "seller_count", "total_freight", "item_count",
            "distinct_product_count", "total_product_volume_cm3", "approval_lag_hours",
            "freight_to_price_ratio", "mean_product_weight_g", "max_product_weight_g",
            "mean_distance_km", "max_distance_km", "promised_days_from_approval"]

before = X_train[log_cols].skew()
after = np.log1p(X_train[log_cols]).skew()
pd.DataFrame({"skew before": before, "skew after log": after}).round(1)

,skew before,skew after log
total_price,11.1,0.3
seller_count,10.7,9.8
total_freight,8.7,1.2
item_count,8.2,4.0
distinct_product_count,7.7,6.2
total_product_volume_cm3,7.7,-0.1
approval_lag_hours,5.2,1.0
freight_to_price_ratio,3.7,1.9
mean_product_weight_g,3.6,0.5
max_product_weight_g,3.6,0.4


## 3.3 Scaling and Encoding

### 3.3a Scaling
All numeric columns are standardized (mean 0, standard deviation 1) with StandardScaler, so that columns measured in very different units are on the same scale (e.g., price, weight in g, distance in km).

**The scaling is applied inside the pipeline (3.4), so it is fit on the training data only.**

In [9]:
from sklearn.preprocessing import StandardScaler

num_cols = X_train_sel.select_dtypes(include="number").columns
scaled = StandardScaler().fit_transform(X_train_sel[num_cols])
pd.DataFrame(scaled, columns=num_cols).agg(["mean", "std"]).round(2)

,promised_days_from_approval,approval_lag_hours,purchase_month,purchase_weekday,purchase_hour,item_count,distinct_product_count,seller_count,total_price,total_freight,mean_product_weight_g,max_product_weight_g,total_product_volume_cm3,mean_distance_km,max_distance_km,same_state_item_share,freight_to_price_ratio
mean,-0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


### 3.3b Encoding
Three columns are text: customer_state (27 values), any_seller_state (22) and primary_product_category (71). The text columns are one-hot encoded, giving each value its own 0/1 column. 

Categories with very few orders (fewer than 100 in the training data) are grouped into a single "infrequent" column instead of getting their own, because there is not enough data for the model to learn much from them.

Finally, the 1,169 missing product categories are filled with "unknown" before encoding, so they become their own category.

**The encoding is applied inside the pipeline (3.4), so it is fit on the training data only.**

In [10]:
cat_cols = X_train_sel.select_dtypes(exclude="number").columns
pd.DataFrame({"different_values": X_train_sel[cat_cols].nunique(),
              "rows_missing": X_train_sel[cat_cols].isna().sum()})

,different_values,rows_missing
customer_state,27,0
any_seller_state,22,0
primary_product_category,71,1169


## 3.4 Pipeline and Leakage Checks

### 3.4a Pipeline
All preprocessing is combined into one scikit-learn pipeline (ColumnTransformer), so every step is fit on the training data only. In cross-validation it is refit on each fold's training part, and the final fitted pipeline is then applied to the test set without refitting.

The steps are:
- Missing values: median for distance and weight, "unknown" for product category, plus a "was missing" flag for the distance columns.
- Log transform (log(1 + x)) on the 13 skewed numeric columns.
- Standardization of all numeric columns.
- One-hot encoding of the three text columns, with rare and unseen categories grouped as "infrequent".

After one-hot encoding the text columns, the pipeline produces 103 features.

In [11]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

dist_cols = ["mean_distance_km", "max_distance_km"]
weight_cols = ["mean_product_weight_g", "max_product_weight_g"]

# Skewed columns with no missing values: logged then scaled
log_cols = ["total_price", "seller_count", "total_freight", "item_count",
            "distinct_product_count", "total_product_volume_cm3",
            "approval_lag_hours", "freight_to_price_ratio", "promised_days_from_approval"]

# Skewed columns with missing values: filled, logged, then scaled
log_fill_cols = dist_cols + weight_cols

# Remaining numerical columns: only scaled
other_num = ["purchase_month", "purchase_weekday", "purchase_hour", "same_state_item_share"]

# Text columns
cat_cols = ["customer_state", "any_seller_state", "primary_product_category"]

# Logged then standardized
log_scale = Pipeline([("log", FunctionTransformer(np.log1p)),
                      ("scale", StandardScaler())])

# Medians filled, logged, then standardized
fill_log_scale = Pipeline([("fill", SimpleImputer(strategy="median")),
                           ("log", FunctionTransformer(np.log1p)),
                           ("scale", StandardScaler())])

# "unknown" filled, then one-hot (rare and unseen values become "infrequent")
cat_pipe = Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="unknown")),
                     ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                              min_frequency=100))])

# Each step applied to its own columns
preprocess = ColumnTransformer([
    ("log_scale", log_scale, log_cols),
    ("fill_log_scale", fill_log_scale, log_fill_cols),
    ("dist_flag", MissingIndicator(features="all"), dist_cols),  # "was missing" flag
    ("scale", StandardScaler(), other_num),
    ("categories", cat_pipe, cat_cols),
], sparse_threshold=0)

# Check: fit on the training data only
X_pre = preprocess.fit_transform(X_train_sel)
print(X_pre.shape)                          
print(np.isnan(X_pre.astype(float)).sum())  

(64038, 103)
0


### 3.4b Leakage Checks
Late delivery predictions are made at the moment an order is approved, so the model can only use information that is already available at that point.
The pipeline is also fit on the training data only so that no validation or test information reaches the fitted steps.

The table below shows which columns are known when the order is approved (no leakage) and which are only known afterwards, during or after delivery (leakage).

| Column | Known at approval? | Reason |
|---|---|---|
| Timing: promised_days_from_approval, approval_lag_hours, purchase_month, purchase_weekday, purchase_hour | Y | Come from the purchase time, approval time and the estimated delivery date, which the customer sees when ordering. |
| Location: customer_state, any_seller_state | Y | Customer and seller locations are known at checkout. |
| Distance: mean_distance_km, max_distance_km, same_state_item_share | Y | Calculated from the customer and seller ZIP codes, both known when the order is placed. |
| Order size and cost: item_count, distinct_product_count, seller_count, total_price, total_freight, freight_to_price_ratio | Y | The items ordered, their prices and the freight are fixed when the order is placed. |
| Product: mean_product_weight_g, max_product_weight_g, total_product_volume_cm3, primary_product_category | Y | Product catalogue attributes that exist before the order. |
| customer_zip_code_prefix, approval_month | Y (dropped) | Known at approval, but dropped for being too detailed (13,584 values) and nearly identical to purchase_month. |
| order_status, order_delivered_carrier_date, order_delivered_customer_date, review columns | N | Only known after approval, during or after delivery. |
| is_late | N | The target, which is only known after delivery. |
